# FLUPS: despike, average, t0 and PCF corrections

Example on 240619, runs 05 to 12 (ZsOMe in hexane). It reproduces the lab's
`240619_OMeZs_sig_U.txt` and `240619_OMeZs_sig_F.txt` (check at the end).

Steps: read, convert pixels to wavelength, despike, subtract the background, average,
shift by the t0 curve, correct with the PCF, convert to the fluorescence wavelength.

Not shown: shot rejection and t0 picking, which are interactive (they will live in `pyspan`).
The t0 curve (measured on PBBO) and the PCF are loaded from files.

Every value marked **choice** is a scientific decision copied from the lab's notebooks
(`run0512.ipynb`, `240619_OMeZs.ipynb`): check it for your own data.

In [ ]:
from pathlib import Path

import numpy as np

from edyx.fana import despike_median, fluo_wl, pcf_correct
from edyx.io import load_txt, stack_traces, write_ta_dat
from edyx.plot import plot_flups, plot_maps
from edyx.tana import avg_scans, shift_trace, subtract_pre_t0
from edyx.utils import between

## Parameters

In [ ]:
data_dir = Path(r"C:\work\data\flups\2024\240619")
runs = range(5, 13)
fnames = [f"240619_run{i:02d}.txt" for i in runs]

t_bg = -1.4             # ps, choice: delays t < t_bg are background, per scan
despike_size = (15, 5)  # choice: median filter size (pixels, delays)
t_pre = -1.3            # ps, choice: pre-t0 subtraction of the average
t0_file = data_dir / "240619_PBBO_t0_curve.txt"  # columns: wl_u, t0
t0_offset = -0.160      # ps, choice: residual t0, from the Raman line (fit not shown)
pcf_file = data_dir / "240619_pcf_final.npz"

out_dir = Path("output")
out_root = "240619_OMeZs"

## Read

Legacy `.txt` files (delay x pixel). `load_txt` returns the standard layout, (wl, delay) with ascending delays.
These old files store the pixel number in place of the wavelength: the calibration below is the one of
that day (the new detector writes the wavelength directly).

In [ ]:
t, pix, sig = stack_traces(load_txt(data_dir / fn) for fn in fnames)
wl_u = (pix - 1) * 0.123 + 244.284  # nm, upconverted wavelength (calibration of 240619)
sig.shape  # (file, wl, delay)

## Despike

Threshold, **choice**: twice the 5-95 percentile span of the background (before `t_bg`, pump scatter excluded).

In [ ]:
background = sig[..., t < t_bg][:, ~between(pix, 615, 681)]  # excludes the pump scatter (pixel numbers)
lo, hi = np.percentile(background, [5, 95])
threshold = 2 * (hi - lo)
print("threshold:", threshold)

despiked = np.array([despike_median(s, despike_size, threshold=threshold) for s in sig])

In [ ]:
i = 0  # compare one scan
fig, axes = plot_maps(t, wl_u, [sig[i], despiked[i]], titles=["raw", "despiked"],
                      diverging=False, vmin=0, vmax=200)
fig, axes = plot_maps(t, wl_u, [sig[i] - despiked[i]], titles=["removed"])

## Background and average

Masking, **choice**: rows 282-283 of the last scan had a cosmic in the background.
The OMeZs product averages the "early" scans only (**choice**, the sample degrades):
`avg_scans(scans)` over all scans gives the lab's `240619_run0512.txt`.

In [ ]:
scans = np.ma.masked_array(subtract_pre_t0(t, despiked, t_bg))
scans[-1, [282, 283], :] = np.ma.masked

early = np.arange(len(fnames)) % 4 < 2  # runs 05, 06, 09, 10
avg = avg_scans(scans[early])
avg = subtract_pre_t0(t, avg, t_pre)

fig, axes = plot_maps(t, wl_u, [avg], diverging=False, vmin=0, vmax=50)

## Time-zero correction

In [ ]:
t0_wl, t0 = np.loadtxt(t0_file).T
assert np.allclose(t0_wl, wl_u), "t0 curve sampled on other wavelengths"

t_s, shifted = shift_trace(t, avg, t0)
t_s = t_s - t0_offset

fig, axes = plot_maps(t, wl_u, [avg], diverging=False, vmin=-10, vmax=50)
axes[0, 0].plot(t0, wl_u, color="r", lw=0.5)
fig, axes = plot_maps(t_s, wl_u, [shifted], diverging=False, vmin=-10, vmax=50)

## PCF and fluorescence wavelength

PCF, **choice**: undefined values (NaN, inf) are replaced by the maximum.

Jacobian, **choice**: the signal is per unit of `wl_u`; it is converted to per unit of `wl_f`
with the numerical derivative, as in the lab's notebook.

In [ ]:
pcf_npz = np.load(pcf_file)
pcf_wl, pcf = pcf_npz["wl"], pcf_npz["pcf"].copy()
pcf[~np.isfinite(pcf)] = np.nanmax(pcf)

sig_u = pcf_correct(wl_u, shifted, pcf_wl, pcf)

wl_f = fluo_wl(wl_u)  # gate 1340 nm
sig_f = sig_u * (np.gradient(wl_u) / np.gradient(wl_f))[:, np.newaxis]

In [ ]:
art = plot_flups(t_s, wl_f, sig_f, vmin=-1, vmax=70, scatter_mask=~between(wl_f, 421, 433))
ax = art[0].axes
ax.set_ylim(355, 650)
ax.set_xlabel("delay / ps")
ax.set_ylabel(r"$\lambda_\mathrm{F}$ / nm")

## Save

In [ ]:
out_dir.mkdir(exist_ok=True)
header = (f"runs: {list(runs)}, early scans: {early.tolist()}\n"
          f"t0 curve: {t0_file.name}, offset {t0_offset} ps\npcf: {pcf_file.name}")
write_ta_dat(out_dir / f"{out_root}_sig_U.dat", t_s, wl_u, sig_u, header=header)
write_ta_dat(out_dir / f"{out_root}_sig_F.dat", t_s, wl_f, sig_f, header=header)

## Check against the lab's output

The lab's files are in the legacy `.txt` layout, written with 6 significant digits.

In [ ]:
for fn, wl_, s in [("sig_U", wl_u, sig_u), ("sig_F", wl_f, sig_f)]:
    t_ref, wl_ref, ref = load_txt(data_dir / f"{out_root}_{fn}.txt")
    assert np.allclose(t_s, t_ref) and np.allclose(wl_, wl_ref)
    print(fn, "max relative difference:", np.max(np.abs(s - ref)) / np.max(np.abs(ref)))
    assert np.ma.allclose(s, ref, rtol=1e-5, atol=1e-3)